# Figure: learned vs. median-heuristic bandwidth

Loads `results/svgd/bandwidth_learning.json` (`bench/svgd/bandwidth_learning_experiment.py`). Never recomputes.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt

_here = Path.cwd()
_candidates = [_here / "results" / "svgd", _here.parents[1] / "results" / "svgd"]
RESULTS = next((c for c in _candidates if c.exists()), _candidates[0])
FIGDIR = RESULTS.parents[1] / "examples" / "differentiable_paper" / "figures"
FIGDIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 120, "font.size": 11, "axes.grid": True, "grid.alpha": 0.3,
    "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False,
})
PALETTE = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3"]
BASELINE = "#8C8C8C"


In [ ]:
data = json.loads((RESULTS / "bandwidth_learning.json").read_text())
meta = data["metadata"]
print(f"device={meta['device_kind']} jax={meta['jax_version']}")


In [ ]:
fig, (ax_h, ax_s) = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
# Left: bandwidth + loss trajectory.
steps = range(len(data["h_history"]))
ax_h.plot(steps, data["h_history"], "-", color=PALETTE[0], label="bandwidth h")
ax_h.axhline(data["h_median"], color=BASELINE, ls=":", label="median heuristic")
ax_h.set_xlabel("outer step"); ax_h.set_ylabel("bandwidth h")
ax_h.set_title("Learned bandwidth"); ax_h.legend()
# Right: dispersion per dimension, target vs median-h vs learned-h.
dims = range(len(data["target_std"]))
w = 0.25
ax_s.bar([d - w for d in dims], data["target_std"], w, label="target", color=BASELINE)
ax_s.bar(list(dims), data["std_median_heuristic"], w, label="median-h", color=PALETTE[1])
ax_s.bar([d + w for d in dims], data["std_learned"], w, label="learned-h", color=PALETTE[2])
ax_s.set_xlabel("dimension"); ax_s.set_ylabel("particle std")
ax_s.set_title("Dispersion: learned vs. median heuristic"); ax_s.legend()
ax_s.set_xticks(list(dims))
out = FIGDIR / "fig_svgd_bandwidth_learning.pdf"
fig.savefig(out); fig.savefig(out.with_suffix(".png"))
print("wrote", out)
